# QC: IRF по месяцам — озеро vs Excel

Не запускает помесячный цикл `final_script_2`.

**Озеро:** `ods_alpha.scd1_trx` (SA / S01) + `ods_alpha.scd1_trx_int.n_amt_fee`.
Дата — `scd1_trx.d_trx_orig` (`trx_int` даты не имеет).
Это тот же периметр, из которого в `final_df` собирается `int_component`.

**Excel:** колонка «Комиссия МПС (IRF, ₽)» в месячных файлах `DATA_DIR`.
Августа в референсах обычно нет — будет только озеро.

Смотрите `int_coverage_pct` и `|IRF|` к предыдущему месяцу: провал покрытия при живом `trx_cnt` = дыра в `scd1_trx_int`, не «бизнес упал».

Костыль `run_august_irf_impute` подставляет **июль → август**. Если в озере дырявый июль, костыль **не** включать.

## Перед письмом в озеро (ячейки в конце)

Нужно отличить «нет IRF в `scd1_trx_int`» от «не те колонки / не тот джойн / пали транзакции».

| # | Проверка | Если дыра настоящая |
|---|---|---|
| 1 | `DESCRIBE scd1_trx_int` — есть `n_trx`, `n_amt_fee` | таблица и поле fee те же, что в витрине |
| 2 | Пересчёт `trx_int` без джойна: сколько строк, fee, deleted | таблица не пустая целиком — или наоборот пустая |
| 3 | Джойн `CAST(n_trx AS STRING)` + `ods_deleted_flg` | те же дыры, что в первом probe — не тип ключа |
| 4 | Контроль `scd1_trx_acq.n_amt_tax` по тем же trx | комиссия жива, пропал только IRF |
| 5 | 30 `n_trx` из «мёртвого» месяца → lookup в `trx_int` | 0 строк = ключей в таблице IRF нет |
| 6 | Опционально: `SUM(int_component)` старого `final_df` | раньше в витрине IRF был, сейчас в ODS нет |

В конце — вердикт и черновик письма. Не писать коллегам, пока вердикт не `дыра trx_int`.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
period_start = '2026-01-01'
period_end_exclusive = '2026-09-01'

excel_header = 0
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-07': -1,
}
excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
    '2026-07': DATA_DIR / '07_Июль_2026.xlsx',
}

IRF_COL_NEEDLES = [
    'Комиссия МПС (IRF, ₽)',
    'Комиссия МПС (IRF, р)',
    'Комиссия МПС (IRF, руб)',
    'Комиссия МПС (IRF)',
    'IRF',
]
TRX_CNT_NEEDLES = ['Количество операций', 'Количеств операций', 'trx_cnt']
TRX_SUM_NEEDLES = ['Сумма операций', 'Сумма опреаций', 'trx_sum']

run_invalidate = True
run_schema_proof = True
run_int_census = True
run_cast_join_proof = True
run_acq_control = True
run_point_lookup = True
run_checkpoint_compare = True
lookup_sample_n = 30
coverage_hole_pct = 5.0

output_xlsx = DATA_DIR / 'qc_irf_lake_vs_excel_2026_01_2026_08.xlsx'
letter_md_path = DATA_DIR / 'qc_irf_lake_letter_draft.md'
checkpoint_csv_candidates = [
    DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_08.csv',
]

print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
for ym, p in excel_reference_by_month.items():
    print(f'  {ym}: exists={p.exists()}  {p.name}')

In [ ]:
def pick_col(columns, needles):
    ranked = []
    for col in columns:
        low = str(col).lower().replace('\n', ' ').strip()
        compact = re.sub(r'[^a-zа-я0-9]+', '', low)
        best = None
        for i, n in enumerate(needles):
            nlow = n.lower().strip()
            ncompact = re.sub(r'[^a-zа-я0-9]+', '', nlow)
            if low == nlow or compact == ncompact:
                score = (0, i)
            elif nlow in low or (ncompact and ncompact in compact):
                score = (1, i)
            else:
                continue
            if best is None or score < best:
                best = score
        if best is not None:
            ranked.append((best, col))
    if not ranked:
        return None
    ranked.sort(key=lambda x: x[0])
    return ranked[0][1]


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def resolve_excel_header(path, header):
    if header != -1:
        return header
    raw = pd.read_excel(path, header=None, nrows=12)
    markers = ('аур', 'амортизац', 'фин. рез', 'фин.рез', 'irf', 'комиссия мпс')
    for i, row in raw.iterrows():
        cells = ' '.join(str(x).lower().replace('\n', ' ') for x in row.tolist() if pd.notna(x))
        if any(m in cells for m in markers):
            print(f'  header=-1 → {i} ({path.name})')
            return i
    print(f'  header=-1 не найден, пробуем 0 ({path.name})')
    return 0


def load_excel_irf(path, header):
    hdr = resolve_excel_header(path, header)
    ex = pd.read_excel(path, header=hdr)
    irf_col = pick_col(ex.columns, IRF_COL_NEEDLES)
    if irf_col is None:
        raise ValueError(f'Нет колонки IRF в {path.name}. Колонки: {list(ex.columns)}')
    trx_cnt_col = pick_col(ex.columns, TRX_CNT_NEEDLES)
    trx_sum_col = pick_col(ex.columns, TRX_SUM_NEEDLES)
    irf = to_num_series(ex[irf_col])
    return {
        'irf_col': irf_col,
        'irf_excel': float(irf.fillna(0).sum()),
        'irf_excel_abs': float(irf.abs().fillna(0).sum()),
        'trx_cnt_excel': float(to_num_series(ex[trx_cnt_col]).fillna(0).sum()) if trx_cnt_col else np.nan,
        'trx_sum_excel': float(to_num_series(ex[trx_sum_col]).fillna(0).sum()) if trx_sum_col else np.nan,
        'excel_rows': int(len(ex)),
    }


def fetch_df(sql, label='query'):
    print(f'[{pd.Timestamp.now().strftime("%H:%M:%S")}] {label}')
    with imp:
        try:
            imp.execute('set MEM_LIMIT=8g')
        except Exception:
            pass
        out = imp.fetch(sql)
    if out is None:
        return pd.DataFrame()
    return out


def colset(df):
    return {str(c).strip().lower() for c in df.columns}


def has_col(df, name):
    return name.lower() in colset(df)


def pick_desc_col(desc, needles):
    if desc is None or desc.empty:
        return None
    name_col = desc.columns[0]
    names = desc[name_col].astype(str)
    low = names.str.lower()
    for n in needles:
        hit = names[low == n.lower()]
        if len(hit):
            return str(hit.iloc[0])
    for n in needles:
        hit = names[low.str.contains(n.lower(), na=False)]
        if len(hit):
            return str(hit.iloc[0])
    return None

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connected')

if run_invalidate:
    with imp:
        for t in ('ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_int', 'ods_alpha.scd1_trx_acq'):
            try:
                imp.execute(f'invalidate metadata {t}')
                imp.execute(f'refresh {t}')
                print('[invalidate ok]', t)
            except Exception as exc:
                print('[invalidate fail]', t, type(exc).__name__)

In [ ]:
irf_probe_sql = f'''
SELECT
  cast(trunc(to_date(to_timestamp(t.d_trx_orig, 'yyyy-MM-dd HH:mm:ss')), 'MM') AS string) AS trx_month,
  count(*) AS trx_cnt,
  count(ti.n_trx) AS with_int_row,
  sum(cast(ti.n_amt_fee AS double)) AS sum_n_amt_fee,
  sum(abs(cast(ti.n_amt_fee AS double))) AS sum_n_amt_fee_abs,
  sum(CASE WHEN ti.n_trx IS NULL THEN 1 ELSE 0 END) AS trx_without_int
FROM ods_alpha.scd1_trx t
LEFT JOIN ods_alpha.scd1_trx_int ti ON ti.n_trx = t.n_trx
WHERE t.c_trx_class = 'SA'
  AND t.c_trx_type = 'S01'
  AND coalesce(t.cf_trx_stat, '') <> 'R'
  AND t.c_nter IS NOT NULL
  AND t.d_trx_orig >= '{period_start}'
  AND t.d_trx_orig <  '{period_end_exclusive}'
GROUP BY 1
ORDER BY 1
'''

print('Lake IRF', period_start, '…', period_end_exclusive)
with imp:
    try:
        imp.execute('set MEM_LIMIT=8g')
    except Exception:
        pass
    lake = imp.fetch(irf_probe_sql)

if lake is None or lake.empty:
    raise RuntimeError('Пустой probe озера')

lake['trx_month'] = lake['trx_month'].astype(str).str[:7]
for c in ('trx_cnt', 'with_int_row', 'sum_n_amt_fee', 'sum_n_amt_fee_abs', 'trx_without_int'):
    lake[c] = pd.to_numeric(lake[c], errors='coerce')
lake['int_coverage_pct'] = np.where(
    lake['trx_cnt'] > 0,
    100.0 * lake['with_int_row'] / lake['trx_cnt'],
    np.nan,
)
print('=== Озеро ===')
display(lake)

In [ ]:
excel_rows = []
for ym, path in excel_reference_by_month.items():
    hdr = excel_header_by_month.get(ym, excel_header)
    rec = {'trx_month': ym, 'excel_path': path.name, 'excel_exists': path.exists()}
    if not path.exists():
        rec['excel_error'] = 'file missing'
        excel_rows.append(rec)
        continue
    try:
        rec.update(load_excel_irf(path, hdr))
        rec['excel_error'] = ''
        print(f'OK {ym}: IRF={rec["irf_excel"]:,.2f}  col={rec["irf_col"]}')
    except Exception as exc:
        rec['excel_error'] = f'{type(exc).__name__}: {exc}'
        print('FAIL', ym, rec['excel_error'])
    excel_rows.append(rec)

excel_df = pd.DataFrame(excel_rows)
print('=== Excel ===')
display(excel_df)

In [ ]:
cmp = lake.merge(
    excel_df.drop(columns=['excel_path'], errors='ignore'),
    on='trx_month',
    how='outer',
)
cmp = cmp.sort_values('trx_month').reset_index(drop=True)

cmp['irf_lake_minus_excel'] = cmp['sum_n_amt_fee'] - cmp['irf_excel']
cmp['irf_abs_lake_minus_excel'] = cmp['sum_n_amt_fee_abs'] - cmp['irf_excel_abs']
cmp['irf_vs_excel_pct'] = np.where(
    cmp['irf_excel'].abs() > 1,
    100.0 * cmp['irf_lake_minus_excel'] / cmp['irf_excel'].abs(),
    np.nan,
)
prev_fee = cmp['sum_n_amt_fee_abs'].shift(1)
prev_trx = cmp['trx_cnt'].shift(1)
cmp['lake_irf_abs_mom'] = np.where(prev_fee.abs() > 1, cmp['sum_n_amt_fee_abs'] / prev_fee, np.nan)
cmp['lake_trx_mom'] = np.where(prev_trx > 0, cmp['trx_cnt'] / prev_trx, np.nan)

def _flag(r):
    if pd.isna(r.get('trx_cnt')):
        return 'нет месяца в озере'
    cov = r.get('int_coverage_pct')
    if pd.notna(cov) and cov < 5:
        return 'дыра озера (почти нет trx_int)'
    if pd.notna(cov) and cov < 50:
        return 'озеро неполное'
    if pd.isna(r.get('irf_excel')):
        return 'нет Excel'
    if pd.notna(r.get('irf_vs_excel_pct')) and abs(r['irf_vs_excel_pct']) > 30:
        return 'большое расхождение с Excel'
    return 'ок'

cmp['flag'] = cmp.apply(_flag, axis=1)

show = [
    'trx_month', 'trx_cnt', 'with_int_row', 'int_coverage_pct', 'trx_without_int',
    'sum_n_amt_fee', 'sum_n_amt_fee_abs', 'lake_trx_mom', 'lake_irf_abs_mom',
    'irf_excel', 'irf_excel_abs', 'irf_lake_minus_excel', 'irf_vs_excel_pct',
    'trx_cnt_excel', 'flag', 'excel_error',
]
show = [c for c in show if c in cmp.columns]
print('=== Сводка IRF: озеро vs Excel ===')
display(cmp[show])

print('\nКак читать:')
print('- trx живой, coverage < 5% → в scd1_trx_int нет месяца, не падение бизнеса')
print('- костыль июль→август имеет смысл, только если июль coverage высокий, август дырявый')
print('- знак: в Excel IRF часто «−», в озере как в источнике; смотрите и raw, и abs')

try:
    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as w:
        lake.to_excel(w, sheet_name='lake', index=False)
        excel_df.to_excel(w, sheet_name='excel', index=False)
        cmp.to_excel(w, sheet_name='compare', index=False)
    print('Saved', output_xlsx)
except Exception as exc:
    print('Save xlsx failed:', type(exc).__name__, exc)

## Доказательства для письма: IRF нет в `scd1_trx_int`

Запускать после сводки озеро vs Excel. Цель — закрыть возражения «не та колонка», «не тот ключ», «пропали все транзакции».

Ожидание при настоящей дыре:
- `n_amt_fee` есть в схеме;
- SA/S01 `trx` за месяц живы (~2 млн);
- тех же `n_trx` нет в `trx_int` (coverage ~0);
- `trx_acq.n_amt_tax` за тот же месяц жив;
- точечный lookup 30 ключей из января/апреля/июля → 0 строк в `trx_int`.

In [ ]:
# 1) Schema: та ли таблица и то ли поле fee
int_desc = pd.DataFrame()
trx_desc = pd.DataFrame()
if not run_schema_proof:
    print('SKIP schema proof')
else:
    int_desc = fetch_df('DESCRIBE ods_alpha.scd1_trx_int', 'DESCRIBE scd1_trx_int')
    trx_desc = fetch_df('DESCRIBE ods_alpha.scd1_trx', 'DESCRIBE scd1_trx')
    print('=== scd1_trx_int ===')
    display(int_desc)
    print('=== scd1_trx (ключ / дата) ===')
    name_col = trx_desc.columns[0]
    trx_keep = trx_desc[trx_desc[name_col].astype(str).str.lower().isin(
        ['n_trx', 'd_trx_orig', 'c_trx_class', 'c_trx_type', 'cf_trx_stat', 'c_nter', 'ods_deleted_flg']
    )]
    display(trx_keep if len(trx_keep) else trx_desc.head(20))

    int_names = set(int_desc[int_desc.columns[0]].astype(str).str.lower())
    need_int = {'n_trx', 'n_amt_fee'}
    missing = sorted(need_int - int_names)
    print('trx_int has n_trx / n_amt_fee:', not missing, '| missing=', missing)
    extra_dates = [
        str(x) for x in int_desc[int_desc.columns[0]]
        if any(k in str(x).lower() for k in ('d_', 'date', 'setl', 'load', 'valid', 'upd'))
    ]
    print('trx_int date-like columns:', extra_dates or 'none')
    if missing:
        raise RuntimeError('В scd1_trx_int нет ожидаемых колонок IRF: ' + ', '.join(missing))

In [ ]:
# 2) Census trx_int без джойна к trx — таблица пустая целиком или нет ключей 2026?
int_census = pd.DataFrame()
int_by_setl = pd.DataFrame()
if not run_int_census:
    print('SKIP trx_int census')
else:
    has_del = False
    has_setl = False
    if int_desc is not None and len(int_desc):
        inames = set(int_desc[int_desc.columns[0]].astype(str).str.lower())
        has_del = 'ods_deleted_flg' in inames
        has_setl = 'd_net_setl' in inames

    del_expr = (
        "SUM(CASE WHEN CAST(ods_deleted_flg AS STRING) IN ('1', 'Y', 'y') THEN 1 ELSE 0 END) AS rows_deleted"
        if has_del else 'CAST(NULL AS BIGINT) AS rows_deleted'
    )
    live_expr = (
        "SUM(CASE WHEN COALESCE(CAST(ods_deleted_flg AS STRING), '0') NOT IN ('1', 'Y', 'y') THEN 1 ELSE 0 END) AS rows_live"
        if has_del else 'COUNT(*) AS rows_live'
    )
    census_sql = f'''
    SELECT
      COUNT(*) AS int_rows_all,
      COUNT(n_trx) AS rows_n_trx_not_null,
      COUNT(DISTINCT CAST(n_trx AS STRING)) AS n_trx_distinct,
      SUM(CASE WHEN n_amt_fee IS NULL THEN 1 ELSE 0 END) AS fee_null,
      SUM(CASE WHEN n_amt_fee IS NOT NULL THEN 1 ELSE 0 END) AS fee_not_null,
      SUM(CAST(n_amt_fee AS DOUBLE)) AS fee_sum_all,
      SUM(ABS(CAST(n_amt_fee AS DOUBLE))) AS fee_abs_all,
      {del_expr},
      {live_expr}
    FROM ods_alpha.scd1_trx_int
    '''
    int_census = fetch_df(census_sql, 'trx_int census (full table)')
    print('=== trx_int целиком (без фильтра месяца) ===')
    display(int_census)

    if has_setl:
        int_by_setl = fetch_df('''
        SELECT
          CAST(TRUNC(CAST(d_net_setl AS TIMESTAMP), 'MM') AS STRING) AS setl_month,
          COUNT(*) AS int_rows,
          COUNT(DISTINCT CAST(n_trx AS STRING)) AS n_trx_distinct,
          SUM(CAST(n_amt_fee AS DOUBLE)) AS fee_sum
        FROM ods_alpha.scd1_trx_int
        WHERE d_net_setl IS NOT NULL
        GROUP BY 1
        ORDER BY 1
        ''', 'trx_int by d_net_setl')
        print('=== trx_int по d_net_setl (если дата в самой таблице IRF) ===')
        display(int_by_setl)
        print('Это не замена месяца витрины (месяц = d_trx_orig). Нужно увидеть: есть ли 2026-01..08 в самой trx_int.')
    else:
        print('Колонки d_net_setl нет — месяц IRF только через join к scd1_trx.d_trx_orig.')

    n_all = float(pd.to_numeric(int_census.iloc[0]['int_rows_all'], errors='coerce') or 0) if len(int_census) else 0
    print('Интерпретация:')
    if n_all < 1000:
        print('  таблица trx_int почти пустая целиком — переливка срезала IRF не только за 2026.')
    else:
        print('  таблица не пустая. Если coverage 2026 ~0, пропали ключи 2026 / не тот n_trx, не «дропнули всю ODS».')

In [ ]:
# 3) Тот же периметр, но CAST(n_trx) + ods_deleted_flg — не срыв из-за типа ключа
lake_cast = pd.DataFrame()
if not run_cast_join_proof:
    print('SKIP CAST join proof')
else:
    int_names = set()
    trx_names = set()
    if int_desc is not None and len(int_desc):
        int_names = set(int_desc[int_desc.columns[0]].astype(str).str.lower())
    if trx_desc is not None and len(trx_desc):
        trx_names = set(trx_desc[trx_desc.columns[0]].astype(str).str.lower())
    int_live = (
        "AND COALESCE(CAST(ti.ods_deleted_flg AS STRING), '0') NOT IN ('1', 'Y', 'y')"
        if 'ods_deleted_flg' in int_names else ''
    )
    trx_live = (
        "AND COALESCE(CAST(t.ods_deleted_flg AS STRING), '0') NOT IN ('1', 'Y', 'y')"
        if 'ods_deleted_flg' in trx_names else ''
    )
    lake_cast = fetch_df(f'''
    SELECT
      CAST(TRUNC(TO_DATE(TO_TIMESTAMP(t.d_trx_orig, 'yyyy-MM-dd HH:mm:ss')), 'MM') AS STRING) AS trx_month,
      COUNT(*) AS trx_cnt,
      COUNT(ti.n_trx) AS with_int_row,
      SUM(CAST(ti.n_amt_fee AS DOUBLE)) AS sum_n_amt_fee,
      SUM(ABS(CAST(ti.n_amt_fee AS DOUBLE))) AS sum_n_amt_fee_abs,
      SUM(CASE WHEN ti.n_trx IS NULL THEN 1 ELSE 0 END) AS trx_without_int
    FROM ods_alpha.scd1_trx t
    LEFT JOIN ods_alpha.scd1_trx_int ti
      ON CAST(ti.n_trx AS STRING) = CAST(t.n_trx AS STRING)
      {int_live}
    WHERE t.c_trx_class = 'SA'
      AND t.c_trx_type = 'S01'
      AND COALESCE(t.cf_trx_stat, '') <> 'R'
      AND t.c_nter IS NOT NULL
      {trx_live}
      AND t.d_trx_orig >= '{period_start}'
      AND t.d_trx_orig <  '{period_end_exclusive}'
    GROUP BY 1
    ORDER BY 1
    ''', 'CAST n_trx + deleted_flg join')
    lake_cast['trx_month'] = lake_cast['trx_month'].astype(str).str[:7]
    for c in ('trx_cnt', 'with_int_row', 'sum_n_amt_fee', 'sum_n_amt_fee_abs', 'trx_without_int'):
        if c in lake_cast.columns:
            lake_cast[c] = pd.to_numeric(lake_cast[c], errors='coerce')
    lake_cast['int_coverage_pct'] = np.where(
        lake_cast['trx_cnt'] > 0,
        100.0 * lake_cast['with_int_row'] / lake_cast['trx_cnt'],
        np.nan,
    )
    print('=== JOIN CAST(n_trx)+live ===')
    display(lake_cast)

    both = lake[['trx_month', 'int_coverage_pct', 'sum_n_amt_fee']].merge(
        lake_cast[['trx_month', 'int_coverage_pct', 'sum_n_amt_fee']],
        on='trx_month',
        how='outer',
        suffixes=('_raw', '_cast'),
    )
    print('=== raw join vs CAST+deleted ===')
    display(both)
    print('Если дыры те же — тип n_trx / deleted не объясняют пустой IRF.')

In [ ]:
# 4) Контроль: комиссия эквайринга (trx_acq.n_amt_tax) по тем же SA/S01
# Если tax жив, а fee мёртв — сломалась только переливка IRF, не слой транзакций.
acq_ctrl = pd.DataFrame()
if not run_acq_control:
    print('SKIP trx_acq control')
else:
    acq_ctrl = fetch_df(f'''
    SELECT
      CAST(TRUNC(TO_DATE(TO_TIMESTAMP(t.d_trx_orig, 'yyyy-MM-dd HH:mm:ss')), 'MM') AS STRING) AS trx_month,
      COUNT(*) AS trx_cnt,
      COUNT(ta.n_trx) AS with_acq_row,
      SUM(CAST(ta.n_amt_tax AS DOUBLE)) AS sum_n_amt_tax,
      SUM(ABS(CAST(ta.n_amt_tax AS DOUBLE))) AS sum_n_amt_tax_abs
    FROM ods_alpha.scd1_trx t
    LEFT JOIN ods_alpha.scd1_trx_acq ta
      ON CAST(ta.n_trx AS STRING) = CAST(t.n_trx AS STRING)
    WHERE t.c_trx_class = 'SA'
      AND t.c_trx_type = 'S01'
      AND COALESCE(t.cf_trx_stat, '') <> 'R'
      AND t.c_nter IS NOT NULL
      AND t.d_trx_orig >= '{period_start}'
      AND t.d_trx_orig <  '{period_end_exclusive}'
    GROUP BY 1
    ORDER BY 1
    ''', 'trx_acq n_amt_tax control')
    acq_ctrl['trx_month'] = acq_ctrl['trx_month'].astype(str).str[:7]
    for c in ('trx_cnt', 'with_acq_row', 'sum_n_amt_tax', 'sum_n_amt_tax_abs'):
        acq_ctrl[c] = pd.to_numeric(acq_ctrl[c], errors='coerce')
    acq_ctrl['acq_coverage_pct'] = np.where(
        acq_ctrl['trx_cnt'] > 0,
        100.0 * acq_ctrl['with_acq_row'] / acq_ctrl['trx_cnt'],
        np.nan,
    )
    side = lake[['trx_month', 'int_coverage_pct', 'sum_n_amt_fee']].merge(
        acq_ctrl[['trx_month', 'acq_coverage_pct', 'sum_n_amt_tax', 'with_acq_row']],
        on='trx_month',
        how='outer',
    )
    print('=== IRF vs комиссия эквайринга (один периметр trx) ===')
    display(side)
    print('Ожидание дыры IRF: acq_coverage десятки %, int_coverage ~0 в тех же месяцах.')

In [ ]:
# 5) Точечный lookup: конкретные n_trx из дырявого месяца есть в trx_int?
lookup_summary = pd.DataFrame()
lookup_hits = pd.DataFrame()
if not run_point_lookup:
    print('SKIP point lookup')
else:
    if 'int_coverage_pct' not in lake.columns:
        raise RuntimeError('Сначала ячейка озера (lake)')
    hole = lake.loc[lake['int_coverage_pct'] < coverage_hole_pct, 'trx_month'].astype(str).tolist()
    alive = lake.loc[lake['int_coverage_pct'] >= 20, 'trx_month'].astype(str).tolist()
    months_lookup = []
    if hole:
        months_lookup.extend(hole[:3])
    if alive:
        months_lookup.append(alive[0])
    if not months_lookup:
        months_lookup = lake['trx_month'].astype(str).head(2).tolist()
    print('Lookup months:', months_lookup, '| hole=', hole, '| alive=', alive)

    rows = []
    hits = []
    for ym in months_lookup:
        y, m = ym.split('-')
        start = f'{y}-{m}-01'
        end_ts = pd.Timestamp(start) + pd.offsets.MonthBegin(1)
        end = end_ts.strftime('%Y-%m-%d')
        keys = fetch_df(f'''
        SELECT CAST(t.n_trx AS STRING) AS n_trx
        FROM ods_alpha.scd1_trx t
        WHERE t.c_trx_class = 'SA'
          AND t.c_trx_type = 'S01'
          AND COALESCE(t.cf_trx_stat, '') <> 'R'
          AND t.c_nter IS NOT NULL
          AND t.d_trx_orig >= '{start}'
          AND t.d_trx_orig <  '{end}'
        LIMIT {int(lookup_sample_n)}
        ''', f'sample n_trx {ym}')
        key_list = [str(x) for x in keys['n_trx'].dropna().tolist()] if keys is not None and len(keys) else []
        if not key_list:
            rows.append({'trx_month': ym, 'sampled': 0, 'found_in_trx_int': 0, 'note': 'no trx sample'})
            continue
        in_list = ','.join("'" + k.replace("'", '') + "'" for k in key_list)
        found = fetch_df(f'''
        SELECT
          CAST(i.n_trx AS STRING) AS n_trx,
          CAST(i.n_amt_fee AS DOUBLE) AS n_amt_fee,
          CAST(i.ods_deleted_flg AS STRING) AS ods_deleted_flg
        FROM ods_alpha.scd1_trx_int i
        WHERE CAST(i.n_trx AS STRING) IN ({in_list})
        ''', f'lookup trx_int {ym}')
        n_found = int(found['n_trx'].nunique()) if found is not None and len(found) else 0
        rec = {
            'trx_month': ym,
            'sampled': len(key_list),
            'found_in_trx_int': n_found,
            'int_rows_returned': 0 if found is None else len(found),
            'sample_n_trx': ', '.join(key_list[:5]),
        }
        rows.append(rec)
        if found is not None and len(found):
            found = found.copy()
            found['trx_month'] = ym
            hits.append(found)
        print(f'  {ym}: sample={len(key_list)} found_in_trx_int={n_found}')

    lookup_summary = pd.DataFrame(rows)
    lookup_hits = pd.concat(hits, ignore_index=True) if hits else pd.DataFrame()
    print('=== Point lookup ===')
    display(lookup_summary)
    if len(lookup_hits):
        print('Найденные строки trx_int (живой месяц / неожиданный hit):')
        display(lookup_hits.head(40))
    print('Для письма: 0 found_in_trx_int на дырявом месяце = этих операций в таблице IRF нет.')

In [ ]:
# 6) Старый final_df (checkpoint/CSV): раньше IRF в витрине был
ckpt_month = pd.DataFrame()
ckpt_path_used = None
if not run_checkpoint_compare:
    print('SKIP checkpoint compare')
else:
    ckpt_path_used = next((p for p in checkpoint_csv_candidates if p.exists()), None)
    if ckpt_path_used is None:
        print('Нет CSV final_df в DATA_DIR. Это не блокер письма — достаточно probe + lookup.')
        for p in checkpoint_csv_candidates:
            print('  missing', p.name)
    else:
        print('Checkpoint:', ckpt_path_used)
        usecols = None
        head = pd.read_csv(ckpt_path_used, nrows=0)
        cols = [c for c in head.columns if str(c).lower() in ('report_month', 'int_component', 'trx_cnt')]
        ckpt = pd.read_csv(ckpt_path_used, usecols=cols)
        ckpt.columns = [str(c).strip().lower() for c in ckpt.columns]
        ckpt['report_month'] = ckpt['report_month'].astype(str).str[:7]
        ckpt['int_component'] = pd.to_numeric(ckpt['int_component'], errors='coerce')
        if 'trx_cnt' in ckpt.columns:
            ckpt['trx_cnt'] = pd.to_numeric(ckpt['trx_cnt'], errors='coerce')
        ckpt_month = ckpt.groupby('report_month', as_index=False).agg(
            int_component_final_df=('int_component', 'sum'),
            trx_cnt_final_df=('trx_cnt', 'sum') if 'trx_cnt' in ckpt.columns else ('int_component', 'size'),
            rows=('int_component', 'size'),
        )
        vs = lake[['trx_month', 'sum_n_amt_fee', 'int_coverage_pct']].merge(
            ckpt_month, left_on='trx_month', right_on='report_month', how='outer'
        )
        print('=== Текущее озеро n_amt_fee vs старый final_df.int_component ===')
        display(vs)
        print('Если |int_component_final_df| десятки млн, а sum_n_amt_fee ~0 — ODS перезалили после сборки витрины.')

In [ ]:
# Вердикт + черновик письма
int_desc = globals().get('int_desc', pd.DataFrame())
int_census = globals().get('int_census', pd.DataFrame())
int_by_setl = globals().get('int_by_setl', pd.DataFrame())
lake_cast = globals().get('lake_cast', pd.DataFrame())
acq_ctrl = globals().get('acq_ctrl', pd.DataFrame())
lookup_summary = globals().get('lookup_summary', pd.DataFrame())
lookup_hits = globals().get('lookup_hits', pd.DataFrame())
ckpt_month = globals().get('ckpt_month', pd.DataFrame())
ckpt_path_used = globals().get('ckpt_path_used', None)

hole_months = []
alive_months = []
if 'lake' in globals() and lake is not None and len(lake):
    hole_months = lake.loc[lake['int_coverage_pct'] < coverage_hole_pct, 'trx_month'].astype(str).tolist()
    alive_months = lake.loc[lake['int_coverage_pct'] >= 20, 'trx_month'].astype(str).tolist()

acq_ok = False
if 'acq_ctrl' in globals() and acq_ctrl is not None and len(acq_ctrl) and hole_months:
    acq_hole = acq_ctrl[acq_ctrl['trx_month'].isin(hole_months)]
    acq_ok = bool(len(acq_hole) and (acq_hole['acq_coverage_pct'] > 20).all())

lookup_zero = False
if 'lookup_summary' in globals() and lookup_summary is not None and len(lookup_summary) and hole_months:
    lk_h = lookup_summary[lookup_summary['trx_month'].isin(hole_months)]
    lookup_zero = bool(len(lk_h) and (lk_h['found_in_trx_int'] == 0).all())

cast_same = False
if 'lake_cast' in globals() and lake_cast is not None and len(lake_cast) and len(lake):
    m = lake[['trx_month', 'int_coverage_pct']].merge(
        lake_cast[['trx_month', 'int_coverage_pct']], on='trx_month', suffixes=('_a', '_b')
    )
    if len(m):
        cast_same = bool(((m['int_coverage_pct_a'] < coverage_hole_pct) == (m['int_coverage_pct_b'] < coverage_hole_pct)).all())

if hole_months and acq_ok and (lookup_zero or not run_point_lookup):
    verdict = 'дыра trx_int'
    verdict_ru = (
        'В ods_alpha.scd1_trx_int нет IRF (n_amt_fee) по ключам SA/S01 за месяцы-дыры. '
        'Слой scd1_trx жив; комиссия trx_acq жива. Это не ошибка колонки витрины.'
    )
elif hole_months:
    verdict = 'дыра вероятна — досмотреть контроли'
    verdict_ru = (
        'Покрытие IRF ~0 при живом trx. Допишите контроль trx_acq и point lookup, '
        'потом отправляйте письмо.'
    )
else:
    verdict = 'дыры по coverage нет'
    verdict_ru = 'Coverage не похож на пустой trx_int. Письмо про «нет IRF» пока не отправлять.'

print('VERDICT:', verdict)
print(verdict_ru)
print('hole_months:', hole_months)
print('alive_months:', alive_months)
print('cast_same_holes:', cast_same, '| acq_alive_on_holes:', acq_ok, '| lookup_zero_on_holes:', lookup_zero)

lines = []
lines.append('Тема: Недогруз / пропадание IRF в ods_alpha.scd1_trx_int (2026)')
lines.append('')
lines.append('Коллеги, добрый день.')
lines.append('')
lines.append(
    'Просим проверить переливку ods_alpha.scd1_trx_int: по периметру эквайринга '
    'SA / S01 (scd1_trx, месяц = d_trx_orig) в таблице IRF нет строк на ключ n_trx.'
)
lines.append('')
lines.append('Как считаем IRF в витрине (без изменения формулы):')
lines.append('  int_component = SUM(scd1_trx_int.n_amt_fee), JOIN n_trx = scd1_trx.n_trx')
lines.append(f'  период: {period_start} … {period_end_exclusive} (не включая конец)')
lines.append('')
lines.append('Что видно сейчас:')
lines.append(f'- месяцы без IRF (coverage < {coverage_hole_pct:.0f}%): {hole_months or "нет"}')
lines.append(f'- месяцы, где IRF частично есть (coverage ≥ 20%): {alive_months or "нет"}')
if 'lake' in globals() and lake is not None and len(lake):
    lines.append('- помесячно (trx / with_int / coverage% / sum n_amt_fee):')
    for _, r in lake.iterrows():
        lines.append(
            f"  {r['trx_month']}: trx={r['trx_cnt']:.0f}, with_int={r['with_int_row']:.0f}, "
            f"cov={r['int_coverage_pct']:.2f}%, fee={r['sum_n_amt_fee']}"
        )
if 'cmp' in globals() and cmp is not None and 'irf_excel' in cmp.columns:
    lines.append('- Excel «Комиссия МПС (IRF)» за те же месяцы (для масштаба, не как источник ODS):')
    for _, r in cmp.iterrows():
        if pd.notna(r.get('irf_excel')):
            lines.append(f"  {r['trx_month']}: excel_irf={r['irf_excel']}, lake_fee={r.get('sum_n_amt_fee')}")
lines.append('')
lines.append('Почему это не ошибка нашей витрины:')
lines.append('- поле IRF в ODS — n_amt_fee в scd1_trx_int (как в raw_trx / DAG);')
lines.append('- CAST(n_trx AS STRING) даёт те же дыры, что обычный join;')
lines.append('- scd1_trx за эти месяцы полный (~2 млн SA/S01);')
lines.append('- scd1_trx_acq.n_amt_tax (комиссия эквайринга) по тем же trx жив;' if acq_ok else '- контроль trx_acq: см. ячейку 4 в тетрадке;')
lines.append('- точечно: выборка n_trx из дырявого месяца в trx_int не находится.' if lookup_zero else '- точечный lookup: см. ячейку 5.')
lines.append('')
lines.append('Просьба:')
lines.append('1) Подтвердить полноту загрузки scd1_trx_int на янв–авг 2026 (ключ n_trx, живые строки, n_amt_fee).')
lines.append('2) Сверить с историческим срезом: раньше за март в этой таблице был IRF порядка десятков млн ₽.')
lines.append('3) Проверить дубли на n_trx там, где строки есть (март ранее: ~667k одинаковых пар).')
lines.append('')
lines.append('Тетрадка: qc_irf_lake_vs_excel.ipynb. Готов приложить xlsx со сводкой.')
lines.append('')
lines.append(f'Вердикт на нашей стороне: {verdict}.')

letter = '\n'.join(lines)
print('=== Черновик письма ===')
print(letter)
try:
    letter_md_path.write_text(letter + '\n', encoding='utf-8')
    print('Saved letter', letter_md_path)
except Exception as exc:
    print('Save letter failed:', type(exc).__name__, exc)

try:
    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as w:
        if 'lake' in globals() and lake is not None:
            lake.to_excel(w, sheet_name='lake', index=False)
        if 'excel_df' in globals() and excel_df is not None:
            excel_df.to_excel(w, sheet_name='excel', index=False)
        if 'cmp' in globals() and cmp is not None:
            cmp.to_excel(w, sheet_name='compare', index=False)
        if int_desc is not None and len(int_desc):
            int_desc.to_excel(w, sheet_name='trx_int_schema', index=False)
        if int_census is not None and len(int_census):
            int_census.to_excel(w, sheet_name='trx_int_census', index=False)
        if int_by_setl is not None and len(int_by_setl):
            int_by_setl.to_excel(w, sheet_name='trx_int_by_setl', index=False)
        if lake_cast is not None and len(lake_cast):
            lake_cast.to_excel(w, sheet_name='lake_cast_join', index=False)
        if acq_ctrl is not None and len(acq_ctrl):
            acq_ctrl.to_excel(w, sheet_name='trx_acq_control', index=False)
        if lookup_summary is not None and len(lookup_summary):
            lookup_summary.to_excel(w, sheet_name='point_lookup', index=False)
        if lookup_hits is not None and len(lookup_hits):
            lookup_hits.to_excel(w, sheet_name='point_lookup_hits', index=False)
        if ckpt_month is not None and len(ckpt_month):
            ckpt_month.to_excel(w, sheet_name='final_df_checkpoint', index=False)
        pd.DataFrame([{
            'verdict': verdict,
            'verdict_ru': verdict_ru,
            'hole_months': ', '.join(hole_months),
            'alive_months': ', '.join(alive_months),
            'cast_same_holes': cast_same,
            'acq_alive_on_holes': acq_ok,
            'lookup_zero_on_holes': lookup_zero,
            'checkpoint': str(ckpt_path_used) if ckpt_path_used else '',
        }]).to_excel(w, sheet_name='verdict', index=False)
    print('Saved', output_xlsx)
except Exception as exc:
    print('Save xlsx failed:', type(exc).__name__, exc)